# PETE 662 Project 2 — Notebook 02: PVT Correlations
**Carlos Buenrostro**

`pvt.py` now has two interchangeable models, chosen with `WellboreCase(pvt_model=...)`:

| `pvt_model` | Class | Source |
|---|---|---|
| `"appendix_b"` (default) | `AppendixBPVT` | Appendix-B curve fits from HW5 / Exam 1 (one oil, no temperature effect, step Z) |
| `"correlations"` | `CorrelationPVT` | Vasquez-Beggs Rs/Bo (Eqs. 3-8–3-15), oil density (3-17/3-18), Beggs-Robinson viscosity (3-19–3-27), Standing pseudocriticals (4-8/4-9), Dranchuk-Abou-Kassem Z (4-5/4-6), Lee-Gonzalez-Eakin gas viscosity (4-23–4-26), liquid mixing (3-29/3-31) |

With `"correlations"`, the bubble point of the **produced** fluid is where Vasquez-Beggs Rs equals the producing GOR
(Sec. 3.2.2), so `case.pb` / `case.Rsb` only feed the IPR.

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pete662_nodal import (WellboreCase, ReservoirInputs, AppendixBPVT, CorrelationPVT,
                           operating_point, vlp_curve, ipr_table, pressure_traverse)
from pete662_nodal.pvt import dak_z

## 1. Tests (now includes `tests/test_pvt.py`)

In [ ]:
import sys
!{sys.executable} -m pytest -q

## 2. Textbook Example 3-3
32 °API oil, γg = 0.71, R_p = 500 scf/STB, WOR = 1.5, at 2000 psia and 150 °F. The book reads the dissolved-gas
gravity γgd = 0.85 from Fig. 3-4 and assumes μw = 1 cp.

In [ ]:
ex33 = WellboreCase(depth=8000, ptf=200, T_surface=150, T_bottom=180, D_in=2.259,
                    GOR=500, WOR=1.5, API=32, gamma_g=0.71, mu_w=1.0, pvt_model="correlations")
pvt = CorrelationPVT(ex33, gamma_gd=0.85)
p, T = 2000, 150
rhoL, muL = pvt.liquid_props(p, T)

pd.DataFrame({
    "Python":   [pvt.Rs(p, T), pvt.Bo(p, T), pvt.Bg(p, T) * 1e3, pvt.z_factor(p, T), pvt.rho_o(p, T),
                 pvt.mu_od(T), pvt.mu_o(p, T), rhoL, muL],
    "Textbook": [369, 1.22, 6.97, 0.805, 47.8, 4.19, 1.16, 55.9, 1.06],
}, index=["Rs (scf/STB)", "Bo (bbl/STB)", "Bg (1e-3 ft3/scf)", "Z (implied by Bg)", "rho_o (lbm/ft3)",
          "mu_od (cp)", "mu_o (cp)", "rho_L (lbm/ft3)", "mu_L (cp)"]).round(4)

## 3. Z-factor check (Example 4-2)
At p_pr = 5.96, T_pr = 1.69 the book prints Z = 0.855 (chart) and "Eq. 4-5 gives Z = 0.89".
The code's DAK gives about 0.915, and an independent correlation (Hall-Yarborough, in `tests/test_pvt.py`) gives 0.913,
so the printed 0.89 looks like a rounding/printing issue. Reported, not tuned.

In [ ]:
print(f"DAK Z at ppr=5.96, Tpr=1.69: {dak_z(5.96, 1.69):.4f}")

## 4. Appendix-B curves vs. correlations (Exam 1 fluid, 180 °F)

In [ ]:
well = WellboreCase(depth=9000, ptf=100, T_surface=150, T_bottom=180, D_in=2.44, GOR=750)
a, c = AppendixBPVT(well), CorrelationPVT(replace(well, pvt_model="correlations"))
T = 180
pp = np.linspace(100, 5000, 120)
print(f"Correlation bubble point of the produced fluid (GOR = {well.GOR:.0f}) at {T} F: {c.bubble_point(T):.0f} psia")

props = [("Rs (scf/STB)", "Rs"), ("Bo (bbl/STB)", "Bo"), ("Oil viscosity (cp)", "mu_o"),
         ("Oil density (lbm/ft3)", "rho_o"), ("Z-factor", "z_factor"), ("Gas viscosity (cp)", "mu_gas")]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (label, m) in zip(axes.flat, props):
    ax.plot(pp, [getattr(a, m)(x, T) for x in pp], "--", label="Appendix-B curves")
    ax.plot(pp, [getattr(c, m)(x, T) for x in pp], label="Correlations")
    ax.set(title=label, xlabel="Pressure (psia)")
    ax.grid(alpha=0.3)
axes.flat[0].legend()
fig.tight_layout(); plt.show()

## 5. Effect on the Exam 1 nodal analysis

In [ ]:
reservoir = ReservoirInputs(reservoir_pressure=3000, bubblepoint_pressure=4336, k_md=13, kro=0.5,
                            h_ft=115, mu_o_cp=0.5, Bo=1.46, re_ft=1490, rw_ft=0.25)
rates = np.linspace(50, 1500, 25)
ipr_df = ipr_table(reservoir.to_ipr(), 200)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ipr_df.qo_stb_day, ipr_df.pwf_psia, "k", lw=2, label="IPR — Vogel")
rows = []
for model, style in [("appendix_b", "--"), ("correlations", "-")]:
    w = replace(well, pvt_model=model)
    for method in ("HB", "BB"):
        v = vlp_curve(rates, w, method)
        r = operating_point(reservoir, w, method)
        line, = ax.plot(v["qo (STB/d)"], v["VLP pwf (psia)"], style, label=f"{method}, {model}")
        if r.status == "flowing":
            ax.scatter([r.q], [r.pwf], color=line.get_color(), zorder=5)
        rows.append({"PVT": model, "VLP": method, "q (STB/d)": r.q, "pwf (psia)": r.pwf})
ax.set(xlabel="Oil rate (STB/d)", ylabel="pwf (psia)", title="Exam 1 — PVT model comparison",
       xlim=(0, None), ylim=(0, None))
ax.grid(alpha=0.3); ax.legend(); plt.show()
pd.DataFrame(rows).round(1)

## 6. Something the Appendix-B curves cannot do: water cut and temperature

In [ ]:
corr = replace(well, pvt_model="correlations")
rows = []
for wor in (0, 0.25, 0.5, 1.0):
    r = operating_point(reservoir, replace(corr, WOR=wor), "HB")
    rows.append({"WOR": wor, "status": r.status, "q oil (STB/d)": r.q, "pwf (psia)": r.pwf})
pd.DataFrame(rows).round(1)